<a href="https://colab.research.google.com/github/KoushalVaswani/Emotion-Classification-NLP/blob/main/Emotion_Classification_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('train.txt' , sep = ';' , header= None , names = ['text' , 'emotions'])

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
unique_emotions = df['emotions'].unique()

In [ ]:
emotion_numbers = {}

In [ ]:
i = 0
for emo in unique_emotions:
    emotion_numbers[emo] = i
    i += 1
df['emotions'] = df['emotions'].map(emotion_numbers)

In [ ]:
df.head()

In [ ]:
df['text'] = df['text'].apply(lambda x: x.lower())

In [ ]:
df.head()

In [ ]:
import string

In [ ]:
def remove_punc(text):
    return text.translate(str.maketrans('' , '' , string.punctuation))

In [ ]:
df['text'] = df['text'].apply(remove_punc)

In [ ]:
def remove_numbers(text):
    new = ""
    for i in text:
        if not i.isdigit():
            new = new + i
    return new

In [ ]:
df['text'] = df['text'].apply(remove_numbers)

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return text

    while '<' in text and '>' in text:
        start = text.find('<')
        end = text.find('>')
        if start < end:
            text = text[:start] + text[end+1:]
        else:
            break
    words = text.split()
    clean_words = []
    for word in words:

        if not (word.startswith('http://') or word.startswith('https://') or word.startswith('www.')):
            clean_words.append(word)


    return " ".join(clean_words).strip()

In [ ]:
df['text'] = df['text'].apply(clean_text)

In [ ]:
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new = new + i
    return new

In [ ]:
df['text'] = df['text'].apply(remove_emojis)

In [ ]:
import nltk

In [ ]:
from nltk.corpus import stopwords

In [ ]:
from nltk.tokenize import word_tokenize

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

In [ ]:
stop_words = set(stopwords.words('english'))

In [ ]:
stop_words

In [ ]:
len(stop_words)

In [ ]:
def remove(txt):
    words = word_tokenize(txt)
    cleaned_text = [word for word in words if word not in stop_words]
    return ' '.join(cleaned_text)


In [ ]:
df['text'] = df['text'].apply(remove)

In [ ]:
df['text']

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
cv = CountVectorizer()

In [ ]:
x = cv.fit_transform(df['text'])

In [ ]:
print("BoW Matrix : ", x.toarray())

In [ ]:
print("BoW Matrix shape : ", x.shape)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
y = df['emotions'].values

In [ ]:
x_train , x_test , y_train , y_test = train_test_split(x , y , test_size=0.2 , random_state = 42)

In [ ]:
print("Training data shape : ",x_train.shape)

In [ ]:
print("Testing data shape : ",x_test.shape)

In [ ]:
from sklearn.naive_bayes import MultinomialNB

In [ ]:
model_nb = MultinomialNB()

In [ ]:
model_nb.fit(x_train , y_train)

In [ ]:
from sklearn.metrics import accuracy_score , classification_report , confusion_matrix

In [ ]:
y_pred = model_nb.predict(x_test)

In [ ]:
accuracy_nb = accuracy_score(y_test , y_pred)

In [ ]:
accuracy_nb

In [ ]:
print(classification_report(y_test , y_pred))

In [ ]:
new_text = ['I am feeling so happy and excited today.']
new_text_bow = cv.transform(new_text)

In [ ]:
pred = model_nb.predict(new_text_bow)

In [ ]:
reverse_emotion_numbers = {v: k for k, v in emotion_numbers.items()}
predicted_emotion = reverse_emotion_numbers[pred[0]]

In [ ]:
print("Predicted emotion : ",predicted_emotion)

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
model_lr = LogisticRegression()

In [ ]:
model_lr.fit(x_train , y_train)

In [ ]:
y_pred_lr = model_lr.predict(x_test)

In [ ]:
accuracy_lr = accuracy_score(y_test , y_pred_lr)

In [ ]:
print("Accuracy of Logistic Regression : ",accuracy_lr)

In [ ]:
print(classification_report(y_test , y_pred_lr))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tfidf = TfidfVectorizer()

In [ ]:
model_lr_tfidf = LogisticRegression()

In [ ]:
x_tfidf = tfidf.fit_transform(df['text'])

In [ ]:
x_train_tfidf , x_test_tfidf , y_train , y_test = train_test_split(x_tfidf , y , test_size=0.2 , random_state = 42)

In [ ]:
model_lr_tfidf.fit(x_train_tfidf , y_train)

In [ ]:
y_pred_tfidf = model_lr_tfidf.predict(x_test_tfidf)

In [ ]:
accuracy_tfidf = accuracy_score(y_test , y_pred_tfidf)

In [ ]:
accuracy_tfidf

In [ ]:
print(classification_report(y_test , y_pred_tfidf))

In [ ]:
my_text = ['I feel so sad and depressed today.']
my_text_tfidf = tfidf.transform(my_text)

In [ ]:
new_pred = model_lr_tfidf.predict(my_text_tfidf)

In [ ]:
predicted_emotion = reverse_emotion_numbers[new_pred[0]]

In [ ]:
print("Predicted emotion : ",predicted_emotion)

In [ ]:
from sklearn.svm import LinearSVC

In [ ]:
model_svm = LinearSVC(random_state = 42 , max_iter = 10000)

In [ ]:
model_svm.fit(x_train , y_train)

In [ ]:
y_pred_svm = model_svm.predict(x_test)

In [ ]:
accuracy_svm = accuracy_score(y_test , y_pred_svm)

In [ ]:
accuracy_svm

In [ ]:
print(classification_report(y_test , y_pred_svm))

In [ ]:
#Trying N-Grams on this dataset

In [ ]:
cv_ngram = CountVectorizer(ngram_range=(1,2))
x_ngram = cv_ngram.fit_transform(df['text'])

In [ ]:
x_train_ng , x_test_ng , y_train , y_test = train_test_split(x_ngram , y , test_size=0.2 , random_state = 42)

In [ ]:
model_svm_ngram = LinearSVC(random_state = 42 , max_iter = 1000)

In [ ]:
model_svm_ngram.fit(x_train_ng , y_train)

In [ ]:
y_pred_ng = model_svm_ngram.predict(x_test_ng)

In [ ]:
accuracy_ng = accuracy_score(y_test , y_pred_ng)

In [ ]:
accuracy_ng

In [ ]:
print(classification_report(y_test , y_pred_ng))

In [ ]:
from sklearn.metrics import confusion_matrix

In [ ]:
cm = confusion_matrix(y_test , y_pred_ng)

In [ ]:
ordered_labels = [reverse_emotion_numbers[i] for i in range(len(reverse_emotion_numbers))]

In [ ]:
plt.figure(figsize = (8 , 6))
sns.heatmap(cm , annot = True , fmt = 'd' , cmap = 'Blues' , xticklabels = ordered_labels , yticklabels = ordered_labels)
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.show()